# PyTorch Tutorial 48: Deploying Models to Edge Devices

**Author:** PyTorch Tutorial Series  
**Date:** 2026  
**Prerequisites:** Notebook 47 (Making Models Smaller) + Notebooks 00-05  
**Time:** ~1.5 hours

---

## What You'll Learn

In Notebook 47, we made models smaller. But a smaller PyTorch model is still **Python code** — and phones don't run Python. We need to *export* our models into formats that edge devices understand.

1. **Why we need to "export" models** — PyTorch is Python, edge devices are not
2. **torch.export** — the new standard for freezing models into a graph
3. **ONNX export** — the universal model format (like PDF for models)
4. **ExecuTorch overview** — PyTorch's official edge solution (1.0 GA Oct 2025)
5. **Which format for which platform** — a practical decision guide

---

## Section 1: The Export Problem

### Why Can't We Just Copy the .py File?

PyTorch models are **Python objects**. When you write `model(x)`, Python runs your `forward()` method line by line. This is great for research, but edge devices (phones, Raspberry Pi, microcontrollers) don't have Python installed.

We need to **convert** our model into a standalone format that contains just the math — no Python needed.

```
The Export Pipeline:

  PyTorch Model (.pt)          Your Python code + weights
        |
        v
  Export / Trace               Freeze the model into a graph of operations
        |
        v
  Edge Format                  Pick one based on your target device:
    |- .onnx                   Universal (runs anywhere)
    |- .pte                    ExecuTorch (mobile/embedded)
    |- .mlpackage              CoreML (Apple devices)
        |
        v
  Device Runtime               Lightweight C++/C engine runs the model
```

### The Three Main Formats

| Format | Best For | Think of it as... |
|--------|----------|-------------------|
| **ONNX** (.onnx) | Any platform | PDF — works everywhere |
| **ExecuTorch** (.pte) | Mobile & embedded | A native app — optimized for the device |
| **CoreML** (.mlpackage) | Apple devices | An Apple-native format |

## Section 2: Setup

Let's import what we need and define a simple model to export.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import os
import tempfile

# Optional imports — these are needed for ONNX export/inference
try:
    import onnx
    print(f"onnx version: {onnx.__version__}")
except ImportError:
    onnx = None
    print("onnx not installed. Run: pip install onnx")

try:
    import onnxruntime as ort
    print(f"onnxruntime version: {ort.__version__}")
except ImportError:
    ort = None
    print("onnxruntime not installed. Run: pip install onnxruntime")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
class SmallCNN(nn.Module):
    """A small CNN for MNIST — same architecture as StudentCNN from Notebook 47."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        """Forward pass: images in, class logits out."""
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


print("SmallCNN defined successfully!")

In [ ]:
# Create model with random weights (good enough for export demos)
model = SmallCNN()
# Always set to inference mode before exporting
model.train(False)

# Create a sample input (1 grayscale 28x28 image)
sample_input = torch.randn(1, 1, 28, 28)

# Quick sanity check
with torch.no_grad():
    output = model(sample_input)

param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count:,}")
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output[0].tolist()[:5]}... (first 5 logits)")

## Section 3: torch.export — The New Standard

### What Is It?

`torch.export.export()` takes your Python model and **freezes** it into a computation graph — a list of math operations with no Python logic left. Think of it like compiling code: the source is human-readable, the compiled version is machine-optimized.

### Why Not TorchScript?

You might see `torch.jit.trace()` or `torch.jit.script()` in older tutorials. These are **TorchScript**, which is being phased out in 2026. `torch.export` is the replacement — it's simpler, more reliable, and works with the full PyTorch ecosystem.

### The Key Idea

```python
# Before: model is Python code — dynamic, flexible, needs Python to run
output = model(input)

# After: exported_program is a frozen graph — static, fast, no Python needed
exported = torch.export.export(model, (input,))
output = exported.module()(input)
```

In [ ]:
# Export the model using torch.export
# The second argument is a tuple of example inputs
exported_program = torch.export.export(model, (sample_input,))

print("Export successful!")
print(f"Type: {type(exported_program)}")
print(f"\nThe exported graph (first 500 chars):")
print(str(exported_program)[:500])

In [ ]:
# Verify: exported model produces the same output as the original
with torch.no_grad():
    original_output = model(sample_input)
    exported_output = exported_program.module()(sample_input)

max_diff = (original_output - exported_output).abs().max().item()
print(f"Original output:  {original_output[0][:5].tolist()}")
print(f"Exported output:  {exported_output[0][:5].tolist()}")
print(f"Max difference:   {max_diff:.10f}")
print(f"Outputs match:    {max_diff < 1e-6}")

### Common Pitfalls with torch.export

Not everything exports cleanly. Here are the common issues:

1. **Dynamic control flow** — `if x.shape[0] > 5:` won't work because the graph is static
2. **Data-dependent control flow** — `if x.sum() > 0:` won't work because values aren't known at export time
3. **Unsupported ops** — some custom operations may not be traceable

**Rule of thumb:** If your `forward()` method is a straight pipeline (conv -> relu -> pool -> linear), it will export perfectly. If it has lots of `if/else` logic, you may need to refactor.

## Section 4: ONNX Export — The Universal Format

### What Is ONNX?

**ONNX** (Open Neural Network Exchange) is a universal format for ML models. Think of it like **PDF for documents** — no matter what app created the PDF, any PDF reader can open it.

Similarly, no matter what framework trained the model (PyTorch, TensorFlow, JAX), any ONNX runtime can run it.

### Why Use ONNX?

- **Runs everywhere**: Windows, Linux, Mac, Android, iOS, Raspberry Pi, web browsers
- **Fast**: ONNX Runtime is highly optimized by Microsoft
- **Easy**: Export is about 5 lines of code
- **Well-supported**: Huge ecosystem of tools and runtimes

In [ ]:
# Export to ONNX — it's surprisingly simple!
onnx_path = os.path.join(tempfile.gettempdir(), "small_cnn.onnx")

torch.onnx.export(
    model,                       # The PyTorch model
    sample_input,                # Example input (for tracing)
    onnx_path,                   # Where to save
    input_names=["image"],       # Name the input
    output_names=["logits"],     # Name the output
    dynamic_axes={               # Allow variable batch size
        "image": {0: "batch"},
        "logits": {0: "batch"},
    },
)

onnx_size = os.path.getsize(onnx_path)
print(f"ONNX model saved to: {onnx_path}")
print(f"ONNX file size: {onnx_size / 1024:.1f} KB")

In [ ]:
# Verify the ONNX model is valid
if onnx is not None:
    try:
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)
        print("ONNX model is valid!")
        print(f"IR version: {onnx_model.ir_version}")
        print(f"Opset version: {onnx_model.opset_import[0].version}")
        print(f"Number of nodes: {len(onnx_model.graph.node)}")
    except Exception as e:
        print(f"ONNX validation failed: {e}")
else:
    print("Skipping validation — install onnx: pip install onnx")

In [ ]:
# Run inference with ONNX Runtime and compare with PyTorch
if ort is not None:
    # Create an ONNX Runtime session
    session = ort.InferenceSession(onnx_path)

    # Run inference (ONNX Runtime uses numpy arrays)
    input_numpy = sample_input.numpy()
    ort_output = session.run(None, {"image": input_numpy})[0]

    # Compare with PyTorch
    with torch.no_grad():
        pytorch_output = model(sample_input).numpy()

    max_diff = abs(pytorch_output - ort_output).max()
    print(f"PyTorch output:      {pytorch_output[0][:5]}")
    print(f"ONNX Runtime output: {ort_output[0][:5]}")
    print(f"Max difference:      {max_diff:.10f}")
    print(f"Outputs match:       {max_diff < 1e-5}")
else:
    print("Skipping — install onnxruntime: pip install onnxruntime")

## Section 5: ExecuTorch — PyTorch's Official Edge Solution

### What Is ExecuTorch?

**ExecuTorch** is PyTorch's official framework for running models on **phones, wearables, and embedded devices**. It hit **1.0 GA in October 2025**, which means it's production-ready.

Think of it this way:
- **ONNX** is like a universal translator — works everywhere, good enough for most cases
- **ExecuTorch** is like a native speaker — deeply optimized for specific hardware

### How It Works

```
PyTorch Model
    |
    v
torch.export.export()        # Step 1: Freeze into a graph (same as above)
    |
    v
ExecuTorch Lowering           # Step 2: Optimize for target hardware
  - XNNPACK delegate (CPU)    #   Choose the right "backend delegate"
  - CoreML delegate (Apple)   #   Each one is optimized for specific hardware
  - QNN delegate (Qualcomm)
  - Vulkan delegate (GPU)
    |
    v
.pte file                     # Step 3: Compact binary format
    |
    v
C++ Runtime on Device          # Step 4: Lightweight runtime (no Python!)
```

### Key Facts

- **Replaces** the deprecated PyTorch Mobile
- **Tiny runtime**: ~300KB for basic inference (vs ~100MB+ for PyTorch)
- **Backed by Meta**: Powers AI features on billions of devices
- **Growing ecosystem**: Samsung, Qualcomm, ARM, Apple all contribute backends

### Important Note

The full ExecuTorch pipeline requires the C++ runtime and platform-specific toolchains. Below we show the **Python export step** (which is what you'd do on your dev machine), with comments explaining the full flow.

In [ ]:
# ExecuTorch Export — Conceptual Pipeline
# The first step (torch.export) works everywhere.
# The ExecuTorch-specific steps require: pip install executorch

# Step 1: Export the model (same torch.export we already know)
exported_for_edge = torch.export.export(model, (sample_input,))
print("Step 1: torch.export - DONE")
print(f"  Exported graph has {len(exported_for_edge.graph.nodes)} nodes")

# Step 2: ExecuTorch lowering (conceptual — requires executorch package)
# This is what the code would look like:
#
#   from executorch.exir import to_edge
#   from executorch.backends.xnnpack.partition import XnnpackPartitioner
#
#   # Convert to edge format
#   edge_program = to_edge(exported_for_edge)
#
#   # Delegate to XNNPACK for optimized CPU execution
#   edge_program = edge_program.to_backend(XnnpackPartitioner())
#
#   # Save as .pte file
#   et_program = edge_program.to_executorch()
#   with open("model.pte", "wb") as f:
#       f.write(et_program.buffer)

print("\nStep 2: ExecuTorch lowering (shown as comments above)")
print("  Requires: pip install executorch")
print("  This would produce a .pte file for on-device inference")

# Step 3: On-device inference uses a C++ runtime
print("\nStep 3: On-device inference would use C++ runtime:")
print('  Module module = Module("model.pte");')
print('  auto output = module.forward(input_tensor);')

## Section 6: Which Format for Which Platform?

Here's your decision guide:

| Platform | Recommended Format | Framework | Notes |
|----------|-------------------|-----------|-------|
| **Android** | ExecuTorch (.pte) or LiteRT | ExecuTorch / LiteRT (formerly TFLite) | ExecuTorch for PyTorch models |
| **iOS** | ExecuTorch (.pte) or CoreML | ExecuTorch / CoreML | CoreML integrates with Xcode |
| **Raspberry Pi** | ONNX | ONNX Runtime | Easy setup, good performance |
| **NVIDIA Jetson** | ONNX or TensorRT | ONNX Runtime / TensorRT | TensorRT for max GPU perf |
| **Web browser** | ONNX | ONNX Runtime Web | Runs in JavaScript! |
| **Any platform** | ONNX | ONNX Runtime | The safe default choice |

### When to Use What

- **Start with ONNX** — it works everywhere and is the easiest
- **Switch to ExecuTorch** when you need maximum performance on mobile
- **Use CoreML** if you're building exclusively for Apple devices and want Xcode integration
- **Use TensorRT** when targeting NVIDIA GPUs and need every last bit of speed

## Section 7: Hands-On — Full Export Pipeline

Let's do a complete end-to-end export: PyTorch model -> ONNX -> inference -> benchmark.

In [ ]:
def benchmark_pytorch(model, input_tensor, num_runs=100):
    """Measure average inference time for a PyTorch model."""
    model.train(False)
    # Warm-up runs
    with torch.no_grad():
        for _ in range(10):
            model(input_tensor)

    # Timed runs
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_runs):
            model(input_tensor)
    elapsed = time.perf_counter() - start

    return elapsed / num_runs * 1000  # Return ms per inference


def benchmark_onnx(session, input_numpy, input_name, num_runs=100):
    """Measure average inference time for an ONNX Runtime session."""
    # Warm-up runs
    for _ in range(10):
        session.run(None, {input_name: input_numpy})

    # Timed runs
    start = time.perf_counter()
    for _ in range(num_runs):
        session.run(None, {input_name: input_numpy})
    elapsed = time.perf_counter() - start

    return elapsed / num_runs * 1000  # Return ms per inference

In [ ]:
# Full pipeline: export and verify
print("=== Full Export Pipeline ===")
print()

# Step 1: Export to ONNX
pipeline_onnx_path = os.path.join(tempfile.gettempdir(), "pipeline_model.onnx")
torch.onnx.export(
    model, sample_input, pipeline_onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
)
file_kb = os.path.getsize(pipeline_onnx_path) / 1024
print(f"Step 1: Exported to ONNX ({file_kb:.1f} KB)")

# Step 2: Verify outputs match
if ort is not None:
    session = ort.InferenceSession(pipeline_onnx_path)
    input_np = sample_input.numpy()

    with torch.no_grad():
        pt_out = model(sample_input).numpy()
    ort_out = session.run(None, {"image": input_np})[0]

    max_diff = abs(pt_out - ort_out).max()
    print(f"Step 2: Outputs verified (max diff: {max_diff:.2e})")
else:
    print("Skipping verification — install onnxruntime")

In [ ]:
# Step 3: Benchmark PyTorch vs ONNX Runtime
if ort is not None:
    pt_time = benchmark_pytorch(model, sample_input)
    ort_time = benchmark_onnx(session, input_np, "image")

    print("Step 3: Benchmark complete")
    print()
    print(f"{'':>20} {'PyTorch':>12} {'ONNX Runtime':>14} {'Speedup':>10}")
    print("-" * 58)
    speedup = pt_time / ort_time
    print(f"{'Inference (ms)':>20} {pt_time:>12.3f} {ort_time:>14.3f} {speedup:>9.2f}x")
    print()
    if ort_time < pt_time:
        print(f"ONNX Runtime is {speedup:.2f}x faster than PyTorch!")
    else:
        print(f"PyTorch is {ort_time / pt_time:.2f}x faster (small models can be faster in PyTorch)")
else:
    print("Skipping benchmark — install onnxruntime")

In [ ]:
# Bonus: test with different batch sizes to see how performance scales
if ort is not None:
    print(f"{'Batch Size':>12} {'PyTorch (ms)':>14} {'ONNX (ms)':>12} {'Speedup':>10}")
    print("-" * 50)

    for batch_size in [1, 4, 16, 64]:
        batch_input = torch.randn(batch_size, 1, 28, 28)
        batch_np = batch_input.numpy()

        pt_ms = benchmark_pytorch(model, batch_input, num_runs=50)
        ort_ms = benchmark_onnx(session, batch_np, "image", num_runs=50)
        speedup = pt_ms / ort_ms

        print(f"{batch_size:>12} {pt_ms:>14.3f} {ort_ms:>12.3f} {speedup:>9.2f}x")

    print("\nONNX Runtime often shows bigger speedups with larger batch sizes.")
else:
    print("Skipping — install onnxruntime: pip install onnxruntime")

## Section 8: Practical Tips

### Before You Export

1. **Always set the model to inference mode** — BatchNorm and Dropout behave differently during training
2. **Test with representative inputs** — use real data, not just random tensors
3. **Simplify your forward() method** — remove debugging code, assertions, and complex branching

### After You Export

4. **Always verify outputs match** — run the same input through both models and compare
5. **Test edge cases** — try different batch sizes, image sizes, etc.
6. **Benchmark on the target device** — laptop speed doesn't predict phone speed

### Common Gotchas

7. **Dynamic shapes**: Use `dynamic_axes` in ONNX export if you need variable batch/sequence length
8. **Operator support**: Not every PyTorch op has an ONNX equivalent — check the [ONNX op list](https://onnx.ai/onnx/operators/)
9. **Numerical precision**: Exported models may have tiny differences (1e-6) — this is normal

### My Recommendation

- **Start with ONNX** — it's the most universal and easiest to debug
- **Use ExecuTorch** when targeting mobile specifically and need the best performance
- **Always combine with Notebook 47 techniques** — compress first, then export

## Section 9: Try It Yourself!

**Exercise 1: Export a Larger Model**

Export a ResNet-18 to ONNX and compare file sizes:
```python
import torchvision.models as models
resnet = models.resnet18(weights=None)
resnet.train(False)
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(resnet, dummy, "resnet18.onnx")
# What's the file size compared to our SmallCNN?
```

**Exercise 2: Different Opset Versions**

ONNX has "opset versions" (like API versions). Try exporting with different ones:
```python
torch.onnx.export(model, sample_input, "model_v11.onnx", opset_version=11)
torch.onnx.export(model, sample_input, "model_v17.onnx", opset_version=17)
# Does the file size change? Does the graph look different?
```

**Exercise 3: Dynamic Batch Size**

Export a model with dynamic batch size and verify it works:
```python
torch.onnx.export(
    model, sample_input, "dynamic.onnx",
    input_names=["input"],
    dynamic_axes={"input": {0: "batch"}}
)
# Then run inference with batch sizes 1, 10, and 100
# Do all three work? Is the output correct for each?
```

## Section 10: Recap

### What We Learned

| Concept | Key Takeaway |
|---------|-------------|
| **The Export Problem** | PyTorch models are Python; edge devices need standalone formats |
| **torch.export** | Freezes your model into a graph — the first step in any export pipeline |
| **ONNX** | Universal format that works everywhere — the "PDF of models" |
| **ExecuTorch** | PyTorch's official mobile/embedded solution (1.0 GA Oct 2025) |
| **Format Choice** | Start with ONNX; use ExecuTorch for mobile-specific optimization |

### The Complete Edge ML Pipeline (Notebooks 46-48)

```
Notebook 46: Understand edge constraints (memory, compute, power)
     |
     v
Notebook 47: Make your model smaller (prune, distill, quantize)
     |
     v
Notebook 48: Export to edge format (ONNX, ExecuTorch)  <-- You are here!
```

**What's next:** In Notebook 49, we'll explore running **small language models on edge devices** — bringing the power of LLMs to phones and embedded systems.

---

*Happy deploying!*